# p493 Reverse Pairs 解析笔记

- 题号：p493
- 题目英文名：Reverse Pairs
- 题目中文名：翻转对
- 题目链接：https://leetcode.cn/problems/reverse-pairs/
- 题型：归并排序 / 分治
- 难度：Hard
- 推荐优先级：低
- Java 原代码位置：`solutions-java/leetcode/p493_reverse_pairs/ReversePairs.java`


## 1. 题目一句话总结

这道题要求统计数组中满足 `i < j` 且 `nums[i] > 2 * nums[j]` 的翻转对数量。

本质上考的是如何把“跨区间配对统计”嵌进归并排序，而不是暴力双重循环。


## 2. 题目理解与约束分析

### 2.1 题目要求

给定数组 `nums`，统计所有满足下面条件的下标对个数：

- `i < j`
- `nums[i] > 2 * nums[j]`

### 2.2 输入与输出

- 输入：整数数组 `nums`
- 输出：翻转对数量
- 返回结果含义：数组中满足条件的配对总数

### 2.3 关键约束

- 数组长度可能较大
- 暴力解法 `O(n^2)` 通常会超时
- 注意 `2 * nums[j]` 可能溢出，需要用更大整数类型比较

### 2.4 示例分析

输入：`[1,3,2,3,1]`

满足条件的翻转对有两组：

- `(3, 1)`
- `(3, 1)`

所以答案是 `2`。


## 3. Java 原代码

```java
package leetcode.p493_reverse_pairs;

public class ReversePairs {

    public static int[] temp = new int[50001];

    public static int reversePairs(int[] nums) {
        if (nums == null || nums.length < 2) {
            return 0;
        }
        return count(nums, 0, nums.length - 1);
    }

    public static int count(int[] nums, int l, int r) {
        if (l == r) {
            return 0;
        }
        int m = l + (r - l) / 2;
        return count(nums, l, m) + count(nums, m + 1, r) + mergeCount(nums, l, m, r);
    }

    public static int mergeCount(int[] arr, int l, int m, int r) {
        int count = 0;
        for (int i = l, j = m + 1; i <= m; i++) {
            while (j <= r && (long) arr[i] > (long) arr[j] * 2) {
                j++;
            }
            count += j - m - 1;
        }

        int index = l;
        int left = l;
        int right = m + 1;
        while (left <= m && right <= r) {
            temp[index++] = arr[left] <= arr[right] ? arr[left++] : arr[right++];
        }
        while (left <= m) {
            temp[index++] = arr[left++];
        }
        while (right <= r) {
            temp[index++] = arr[right++];
        }
        for (index = l; index <= r; index++) {
            arr[index] = temp[index];
        }
        return count;
    }
}
```

Java 文件里还附带了一个 `smallSum` 扩展练习，但本题真正对应 LeetCode 493 的核心是 `reversePairs / count / mergeCount` 这几部分。


## 4. 先从 Java 原方案理解

Java 原方案的关键不是“归并排序”本身，而是把翻转对统计插进了 merge 之前。

对于一次归并：

- 左半段 `[l, m]` 已经有序
- 右半段 `[m+1, r]` 也已经有序

这时对每个左侧元素 `arr[i]`，只需要让右侧指针 `j` 单调向前推进，就能快速找出满足：

```text
arr[i] > 2 * arr[j]
```

的右侧范围长度。统计完之后，再做正常归并。

Java 原解里非常重要的一点是：

```java
(long) arr[i] > (long) arr[j] * 2
```

这说明作者专门防了整数溢出。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最直接就是双重循环，枚举所有 `(i, j)`，检查是否满足条件。

### 5.2 为什么不够好

- 时间复杂度是 `O(n^2)`
- 数据量大时会超时

### 5.3 优化方向

如果左右两边都是有序的，就可以利用单调性让右侧指针只前进不回退。归并排序正好能不断制造这种“左右有序”的局面，所以 Java 原方案自然选择了分治 + 归并统计。


## 6. 核心算法讲解

### 6.1 算法名称

- 归并排序
- 分治统计翻转对

### 6.2 为什么想到这个算法

因为翻转对的难点主要在“跨左右两个区间”的配对，而归并排序天然就把数组拆成左右两半，正好方便在 merge 阶段统计跨区间答案。

### 6.3 关键状态

- `l`、`m`、`r`：当前分治区间边界
- `i`：左半段扫描指针
- `j`：右半段扫描指针
- `temp`：归并辅助数组

### 6.4 步骤拆解

1. 递归统计左半段翻转对数量
2. 递归统计右半段翻转对数量
3. 在 merge 前统计跨左右区间的翻转对数量
4. 再把左右两段正常归并成有序数组
5. 返回三部分数量之和


## 7. 过程演示

以 `nums = [2,4,3,5,1]` 为例，某一层归并时可能出现：

```text
左半段： [2, 4, 3]
右半段： [5, 1]
```

在更深层处理后，两边会先变成有序：

```text
左半段： [2, 3, 4]
右半段： [1, 5]
```

统计跨区间时：

- `2 > 2*1` 不成立
- `3 > 2*1` 成立
- `4 > 2*1` 成立

所以这里会累计两组跨区间翻转对。之后再正常 merge，继续给更上层提供有序结果。


In [ ]:
class Solution:
    def reversePairs(self, nums: list[int]) -> int:
        if nums is None or len(nums) < 2:
            return 0

        temp = [0] * len(nums)

        def count(l: int, r: int) -> int:
            if l == r:
                return 0

            m = l + (r - l) // 2
            return count(l, m) + count(m + 1, r) + merge_count(l, m, r)

        def merge_count(l: int, m: int, r: int) -> int:
            ans = 0
            j = m + 1
            for i in range(l, m + 1):
                while j <= r and nums[i] > 2 * nums[j]:
                    j += 1
                ans += j - (m + 1)

            left, right, idx = l, m + 1, l
            while left <= m and right <= r:
                if nums[left] <= nums[right]:
                    temp[idx] = nums[left]
                    left += 1
                else:
                    temp[idx] = nums[right]
                    right += 1
                idx += 1

            while left <= m:
                temp[idx] = nums[left]
                left += 1
                idx += 1

            while right <= r:
                temp[idx] = nums[right]
                right += 1
                idx += 1

            for idx in range(l, r + 1):
                nums[idx] = temp[idx]

            return ans

        return count(0, len(nums) - 1)


## 8. 代码逐段讲解

### 8.1 `count`

`count(l, r)` 对应 Java 原解里的递归入口。它负责把答案拆成三部分：左边、右边、跨区间。

### 8.2 `merge_count`

这一步先统计，再 merge。统计时 `j` 不会回退，这正是有序带来的效率提升。

### 8.3 为什么能单调推进 `j`

因为左半段是有序的，`i` 越往右，`nums[i]` 越大，所以右侧满足条件的范围不会往回缩。

### 8.4 为什么还要正常 merge

因为上层递归还需要当前区间保持有序，才能继续统计更大的跨区间答案。这一步和 Java 原解完全一致。


## 9. 复杂度分析

- 时间复杂度：`O(n log n)`
- 为什么是这个时间复杂度：每层做一次线性 merge，递归深度是 `log n`
- 空间复杂度：`O(n)`
- 为什么是这个空间复杂度：需要辅助数组 `temp`


## 10. 易错点

1. 先 merge 再统计，破坏了原本左右分区边界。
2. `j` 在每个 `i` 上重新从 `m+1` 开始，虽然能做对，但效率会变差。
3. 忽略乘 2 的溢出风险，Java 原解专门用了 `long`。
4. 把普通逆序对统计逻辑直接套过来，条件其实已经变成了 `nums[i] > 2 * nums[j]`。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会用归并排序来做，因为暴力是 `O(n^2)`，而题目规模通常不允许。
> 归并时左右两半都是有序的，所以我可以在 merge 前先用双指针统计跨区间翻转对数量。
> 对每个左侧元素，右侧指针只单调向前推进，不会回退。
> 统计完再做正常 merge，保证上层递归还能继续利用有序性。
> 所以整体时间复杂度是 `O(n log n)`。


## 12. 实际应用场景

这题可以类比成：统计一个时间序列里“前面数值远大于后面数值”的异常跨度对。

### 具体业务案例：价格异常跳变检测

#### 业务背景

金融或监控系统里，经常要统计某个时刻的数值相比后续时刻是否异常偏大。

#### 输入是什么

输入是一段按时间排列的数值数组。

#### 算法介入点

系统要统计有多少对时刻满足“前值超过后值两倍以上”的异常关系。

#### 输出是什么

输出是满足条件的配对数量。

#### 价值是什么

它解决的是跨时间窗口的大幅度反转关系统计问题。


In [ ]:
print('示例一:', Solution().reversePairs([1, 3, 2, 3, 1]))
print('示例二:', Solution().reversePairs([2, 4, 3, 5, 1]))
print('无翻转对:', Solution().reversePairs([1, 2, 3, 4]))


## 13. Demo 输出说明

- 第一组应输出 `2`，对应题目经典示例。
- 第二组应输出 `3`，说明跨区间统计逻辑正确。
- 第三组应输出 `0`，说明没有满足条件的配对时能正确返回零。


## 14. 一句话复盘

> 这题最关键的不是会写归并，而是像 Java 原解那样把翻转对统计嵌进 merge 之前。
